# Solar Filament Segmentation 2026 -- end-to-end inference pipeline

Two-stage instance segmentation of solar filaments in H-alpha images (MAGFiLO):

1. **Stage-A** -- 5-fold ResNet34 U-Net ensemble with dihedral test-time augmentation
   (identity, left-right, up-down, 180 deg) on 512 px sliding windows -> probability map.
2. **Candidates** -- `prob > 0.4`, 3x3 closing, 8-connected components, keep
   `400 <= area <= 40000` and `mean prob >= 0.7`.
3. **Crop refiner** -- every candidate is redrawn by a `smp.Unet(resnet34)` from a box-shaped
   seed (input `[crop, CLAHE(crop), seed]`, 512x512 crop, threshold 0.5).
4. **Near-limb filter** -- instances whose centroid lies beyond 0.88 of the solar-disk radius
   (disk from an Otsu threshold) are dropped; `Pipeline(..., limb_drop=None)` disables it.
5. **Instance scorer** -- a gradient-boosted regressor (`filament_scorer.py`, model in `models/`) predicts
   every instance's expected matched IoU from inference-time features; instances below the PQ
   break-even (PQ/2 = 0.197) are dropped; `Pipeline(..., scorer=None)` disables it.
6. One mask = one row (`filament_id`, compressed COCO RLE `counts`).

All code is in `filament_pipeline.py` (inference), `filament_scorer.py` (instance scorer) and
`filament_metric.py` (official PQ).
Weights: `unet_fold1..5.pt`, `refiner_best.pt` (Apache-2.0); republished as a Kaggle dataset
shortly before the submission deadline -- see the repository README for the current status
and SHA256 checksums.
No training happens in this notebook; training scripts are in `training/`.

Reported result of this exact pipeline: hold-out PQ 0.4427 (0.4383 without the scorer, 0.4325
also without the near-limb filter; 50 images never used for training or checkpoint selection),
public leaderboard 0.37 (0.36 without the scorer).

## 1. Setup and configuration

In [ ]:
import subprocess, sys
try:
    import segmentation_models_pytorch  # noqa: F401
except ImportError:                     # not preinstalled on Kaggle images
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "segmentation-models-pytorch==0.5.0", "timm==1.0.26"], check=True)

In [ ]:
import os, sys, glob, json, time
import numpy as np, pandas as pd

sys.path.insert(0, ".")          # run from the folder that contains filament_pipeline.py
import filament_pipeline as fp
import filament_metric as fm
import filament_scorer as fs

# --- knobs -------------------------------------------------------------------------
MAX_TEST_IMAGES = None      # None = all 180 test images (GPU: ~1-2 h; CPU: many hours)
EVAL_HOLDOUT_IMAGES = 0     # >0: also score that many hold-out images with the official PQ
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def find_first(root, predicate):
    for r, _, files in os.walk(root):
        for f in files:
            if predicate(r, f):
                return os.path.join(r, f)

# --- locate weights and competition data (Kaggle layout, or set the env vars) ------
INPUT = os.environ.get("FILAMENT_INPUT", "/kaggle/input")
WEIGHTS_DIR = os.environ.get("FILAMENT_WEIGHTS")
fold_paths = sorted(glob.glob(os.path.join(WEIGHTS_DIR, "unet_fold*.pt"))) if WEIGHTS_DIR else sorted(
    p for p in glob.glob(os.path.join(INPUT, "**", "unet_fold*.pt"), recursive=True))
refiner_path = os.path.join(WEIGHTS_DIR, "refiner_best.pt") if WEIGHTS_DIR else find_first(
    INPUT, lambda r, f: f == "refiner_best.pt")
ann_json = find_first(INPUT, lambda r, f: f.endswith(".json") and "nnotation" in f)
train_dir = next((r for r, _, _ in os.walk(INPUT) if os.path.basename(r) == "train_images"), None)
test_dir = next((r for r, _, _ in os.walk(INPUT) if os.path.basename(r) == "test_images"), None)
assert len(fold_paths) == 5 and refiner_path, "weights not found (attach the weights dataset)"
print(len(fold_paths), "stage-A checkpoints |", refiner_path)
print("annotations:", ann_json, "| train:", train_dir, "| test:", test_dir)

## 2. Load models

In [ ]:
import torch
device = fp.default_device()
TAU = None   # None: the released scorer threshold 0.1972 (PQ-optimal); 0.15: the balanced Dice/PQ configuration (report, Table 3)
pipe = fp.Pipeline(fp.load_stage_a(fold_paths, device), fp.load_refiner(refiner_path, device), device,
                   scorer=fs.InstanceScorer.load(tau=TAU))   # scorer=None: the pipeline without the instance scorer
print("device:", device, "| torch", torch.__version__, "| scorer tau", pipe.scorer.tau if pipe.scorer else None)

## 3. (Optional) Score the hold-out with the official PQ

The hold-out is the 50 images selected by `np.random.default_rng(1234).permutation` over the
sorted file names; it was excluded from every training set. Set `EVAL_HOLDOUT_IMAGES` to run it
(the first N of the 50).

In [ ]:
if EVAL_HOLDOUT_IMAGES:
    per_file, dims = fm.load_annotations(ann_json, train_dir)
    names = fm.holdout_names(list(per_file))[:EVAL_HOLDOUT_IMAGES]
    preds = {}
    for k, n in enumerate(names, 1):
        t0 = time.time()
        preds[n] = fp.label_map_to_rles(pipe.segment(fp.read_gray(os.path.join(train_dir, n))))
        print(f"{k}/{len(names)} {n}: {len(preds[n])} instances, {time.time()-t0:.0f}s", flush=True)
    res = fm.evaluate(preds, ann_json, names, train_dir)
    print("PQ", round(res["pq"], 4), {k: res[k] for k in ("tp", "fp", "fn")})
    json.dump({n: v for n, v in res["per_image"].items()},
              open(os.path.join(OUT_DIR, "holdout_eval.json"), "w"), indent=1)

## 4. Predict the test set and write `submission.csv`

In [ ]:
test_files = sorted(f for f in os.listdir(test_dir) if f.lower().endswith((".jpeg", ".jpg")))
if MAX_TEST_IMAGES:
    test_files = test_files[:MAX_TEST_IMAGES]
rows, t_start = [], time.time()
for k, fname in enumerate(test_files, 1):
    stem = fname.rsplit(".", 1)[0]
    rles = fp.label_map_to_rles(pipe.segment(fp.read_gray(os.path.join(test_dir, fname))))
    rows += [{"filament_id": f"{stem}_{i}", "segmentation_rle": r} for i, r in enumerate(rles, 1)]
    if k % 10 == 0 or k == len(test_files):
        print(f"{k}/{len(test_files)}  {time.time()-t_start:.0f}s  rows={len(rows)}", flush=True)
sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
print(fp.validate_submission_rows(rows, {f.rsplit(".", 1)[0] for f in test_files}))

## Notes
* Images without any detected filament produce no rows (the scorer counts their ground-truth
  filaments as false negatives).
* CPU and GPU can differ by tiny floating-point amounts (reduction order); a run on the other
  device may move one borderline instance.
* Rules: the model uses only the provided H-alpha test images at inference and only the
  competition's own annotations for training.